# Making Sense of Data with NumPy, pandas, and Matplotlib

In this workshop we will follow one practical workflow: **load data, inspect it, clean it, ask questions, and communicate the answers with charts**. We will use California census data throughout so that each new tool builds on the previous one.

By the end, you will be able to:

- explain the relationship between a NumPy array, a pandas Series, and a pandas DataFrame;
- select, filter, clean, transform, and summarize tabular data;
- choose and create a few useful chart types with Matplotlib;
- perform a small exploratory data analysis from beginning to end.

## 1. Setup

The conventional aliases `np`, `pd`, and `plt` are used in most Python data projects.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

## 2. NumPy: the numerical foundation

A NumPy **array** stores values of one data type in one or more dimensions. Arrays matter because they support fast, element-wise calculations without writing a Python loop. pandas and many machine-learning libraries build on this idea.

In [ ]:
prices = np.array([10.0, 15.0, 20.0])
quantities = np.array([5, 8, 3])

print("Values:", prices)
print("Shape:", prices.shape)
print("Data type:", prices.dtype)

## Why is NumPy everywhere?
Python arrays are untyped which is usuaully very slow. Because of this basically any libarary that handles big data would use numpy at least SOMEWHERE in its API. We will see this in a bit with pandas and later with pytorch as well.

Usually our job as data scientists is to get data from some text/database into a nice numpy array.
From there we can usually do all the math we need.

### Indexing and slicing

Python starts counting at zero. A single index selects one value; a slice selects a range. The stop position is not included.

In [ ]:
print("First price:", prices[0])
print("First two prices:", prices[:2])

### Vectorized calculations and aggregation

Multiplying the two arrays pairs values by position. Aggregation methods then reduce many values to a useful summary.

In [ ]:
revenue_per_item = prices * quantities

print("Revenue per item:", revenue_per_item)
print("Total revenue:", revenue_per_item.sum())
print("Average price:", prices.mean())

### Boolean masks

A comparison creates `True`/`False` values. Using that result as an index keeps only matching elements. pandas filtering uses the same pattern.

In [ ]:
expensive_mask = prices > 12
print(expensive_mask)
print(prices[expensive_mask])

### Checkpoint 1

Change the quantities or prices above. Predict the total revenue before rerunning the calculation. Then try selecting items with revenue greater than 75.

In [ ]:
# Try your NumPy expression here.


## 3. pandas: labelled, tabular data

A pandas **Series** is similar to a labelled one-dimensional array. A **DataFrame** combines Series into a table: rows are observations and columns are variables. Unlike a NumPy array, different DataFrame columns can have different data types.

In [ ]:
example = pd.DataFrame({
    "item": ["A", "B", "C"],
    "price": prices,
    "quantity": quantities,
})
example

### Moving between pandas and NumPy

pandas can take NumPy arrays as input. You can also use `.to_numpy()` to give a Series or DataFrame's values to code that expects a NumPy array. The array contains the values, but not pandas labels such as column names or the index.

In [ ]:
measurements = np.array([
    [10.0, 5],
    [15.0, 8],
    [20.0, 3],
])

# Give pandas a NumPy array and add meaningful column labels.
measurements_df = pd.DataFrame(measurements, columns=["price", "quantity"])

# Take values from pandas as NumPy arrays.
price_array = measurements_df["price"].to_numpy()
all_values_array = measurements_df.to_numpy()

print(type(price_array), price_array)
print(type(all_values_array), all_values_array.shape)
measurements_df

## 4. Load a real dataset

The California housing dataset is based on the 1990 census. Each row describes a **block group**, not one house. For example, `total_rooms` is the number of rooms across the entire district.

In [ ]:
data_path = Path("california_housing.csv")
if not data_path.exists():
    raise FileNotFoundError("Run this notebook from the '1. Making sense of Big Data' folder.")

housing = pd.read_csv(data_path)
housing.head()

## 5. Inspect before analysing

Start every analysis by asking: How large is the table? What does each row represent? Which columns are numeric or categorical? Are values missing or suspicious?

In [ ]:
print("Rows and columns:", housing.shape)
housing.info()

In [ ]:
housing.describe().round(2)

In [ ]:
housing["ocean_proximity"].value_counts()

`describe()` summarizes numeric columns by default. `value_counts()` is often more informative for a categorical column. Neither replaces understanding what the columns mean.

## 6. Select and filter

Use square brackets to select columns. Use `.loc` for row/column labels and `.iloc` for integer positions.

In [ ]:
housing[["median_income", "median_house_value"]].head()

In [ ]:
print("Label-based selection:")
display(housing.loc[0:2, ["median_income", "median_house_value"]])

print("Position-based selection:")
display(housing.iloc[0:3, 7:9])

Multiple conditions require parentheses. Use `&` for AND, `|` for OR, and `~` for NOT.

In [ ]:
young_near_bay = housing[
    (housing["housing_median_age"] < 10)
    & (housing["ocean_proximity"] == "NEAR BAY")
]

print("Matching districts:", len(young_near_bay))
young_near_bay.head()

### Checkpoint 2

Find districts with a median house value above 350,000 and a median income above 5. How many are there? Display only the income, house value, and ocean-proximity columns.

In [ ]:
# Write your pandas filter here.


## 7. Missing values and data cleaning

Missing data is normal. First measure it, then choose a treatment based on what the value means. Common options are to remove affected rows or fill values with a statistic such as the median.

In [ ]:
housing.isna().sum().sort_values(ascending=False)

In [ ]:
clean_housing = housing.copy()
bedroom_median = clean_housing["total_bedrooms"].median()
clean_housing["total_bedrooms"] = clean_housing["total_bedrooms"].fillna(bedroom_median)

print("Missing values remaining:", clean_housing.isna().sum().sum())

We made a copy so the raw imported data remains unchanged. Median filling is a simple teaching example, not a universally correct choice. In a real project, document the decision and test alternatives.

## 8. Create useful features

Raw totals can be misleading because districts have different sizes. Ratios such as rooms per household make observations more comparable. Column arithmetic is vectorized, just like the NumPy calculation earlier.

In [ ]:
clean_housing["rooms_per_household"] = (
    clean_housing["total_rooms"] / clean_housing["households"]
)
clean_housing["people_per_household"] = (
    clean_housing["population"] / clean_housing["households"]
)

clean_housing[["rooms_per_household", "people_per_household"]].describe().round(2)

Notice the extreme maximum values. Summary statistics help reveal possible outliers, data errors, or unusual cases that deserve investigation.

## 9. Group and aggregate

`groupby()` follows a split-apply-combine pattern: split rows into groups, apply a calculation, and combine the results.

In [ ]:
proximity_summary = (
    clean_housing.groupby("ocean_proximity")
    .agg(
        districts=("median_house_value", "size"),
        average_value=("median_house_value", "mean"),
        average_income=("median_income", "mean"),
    )
    .sort_values("average_value", ascending=False)
)
proximity_summary.round(2)

An association in an aggregated table does not prove that ocean proximity **causes** house prices. Other variables may differ between these groups.

## 10. Matplotlib: communicate with charts

Choose a chart based on the question:

- **Histogram:** What is the distribution of one numeric variable?
- **Bar chart:** How do values compare across categories?
- **Scatter plot:** How are two numeric variables related?
- **Line chart:** How does a value change across an ordered sequence, usually time?

We will use Matplotlib's object-oriented interface: `fig` is the whole canvas and `ax` is one plotting area.

### Distribution: histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(clean_housing["median_house_value"], bins=30, color="#2a6f97", edgecolor="white")
ax.set(
    title="Distribution of median house values",
    xlabel="Median house value (USD)",
    ylabel="Number of districts",
)
plt.show()

The spike at the upper end suggests the target value was capped near $500,000. A chart can expose details that a mean alone hides.

### Category comparison: bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    proximity_summary.index,
    proximity_summary["average_value"],
    color="#61a5c2",
)
ax.set(
    title="Average house value by ocean proximity",
    xlabel="Ocean proximity",
    ylabel="Average median house value (USD)",
)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### Relationship: scatter plot

With many rows, transparent points and a sample reduce overplotting. Setting `random_state` makes the sample reproducible.

In [ ]:
plot_sample = clean_housing.sample(3000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 5))
points = ax.scatter(
    plot_sample["median_income"],
    plot_sample["median_house_value"],
    c=plot_sample["housing_median_age"],
    cmap="viridis",
    alpha=0.35,
    s=18,
)
ax.set(
    title="Income and house value across districts",
    xlabel="Median income (tens of thousands of USD)",
    ylabel="Median house value (USD)",
)
fig.colorbar(points, ax=ax, label="Median housing age")
plt.show()

## 11. Put it together: a small EDA dashboard

Subplots let us compare several views. This is exploratory: each chart should lead to a clearer question, not just decorate the notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(
    clean_housing["rooms_per_household"].clip(upper=10),
    bins=30,
    color="#f4a261",
    edgecolor="white",
)
axes[0].set(
    title="Rooms per household (clipped at 10)",
    xlabel="Rooms per household",
    ylabel="Number of districts",
)

location_plot = axes[1].scatter(
    plot_sample["longitude"],
    plot_sample["latitude"],
    c=plot_sample["median_house_value"],
    cmap="plasma",
    alpha=0.5,
    s=12,
)
axes[1].set(
    title="House values by location",
    xlabel="Longitude",
    ylabel="Latitude",
)
fig.colorbar(location_plot, ax=axes[1], label="Median house value (USD)")
fig.suptitle("A first look at California housing", fontsize=15)
plt.tight_layout()
plt.show()

`clip(upper=10)` changes only the values passed to this chart; it does not remove rows from `clean_housing`. Always disclose transformations that affect a visualization.

## Final challenge

Use the workflow on a question of your own. For example: **Do newer districts tend to have more expensive homes, and does the pattern differ by ocean proximity?**

1. State the question in plain language.
2. Select or create the columns you need.
3. Check missing values and possible outliers.
4. Use filtering or `groupby()` to calculate a summary.
5. Create one appropriate, clearly labelled chart.
6. Write two observations and one limitation.

In [ ]:
# Build your analysis here.


## Takeaways

- NumPy provides efficient arrays, vectorized calculations, masks, and aggregations.
- pandas adds labels and table operations for real-world data.
- Matplotlib turns analysis results into purposeful visual explanations.
- A reliable workflow is iterative: **inspect, clean, transform, summarize, visualize, question, repeat**.
- A chart shows patterns, but context and careful reasoning are needed before making causal claims.